# 04 - Price-trapped districts (Q4)

A district x crop is *price-trapped* when it has <= 2 distinct town locations with valid data within 50 km of its centre AND its average price is < 90% of the state price on the same days.

In [1]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd
from mandipulse import viz
from mandipulse.viz import COMMODITIES, COMMODITY_COLORS, INK, INK_2, MUTED, load
pd.set_option("display.width", 160)
viz.style()

In [2]:
da = load("phase3", "q4_district_access")
summary = load("phase3", "q4_trapped_summary")
summary

,commodity,n_districts,n_trapped,n_thin_access,n_low_price,opp_rate_pct_trapped,opp_rate_pct_others
0,Onion,161,10,45,25,67.2,41.2
1,Potato,151,10,47,25,45.8,31.6
2,Tomato,151,9,46,15,45.3,47.7


In [3]:
viz.style()
fig, axes = plt.subplots(1, 3, figsize=(12, 4.6), sharey=True)
viz.header(fig, "Q4. District access vs price level",
           "Each dot = district x crop. Shaded corner = price-trapped (<= 2 towns within 50 km and price index < 0.9)")
rng = np.random.default_rng(0)
for ax, c in zip(axes, COMMODITIES, strict=True):
    g = da[da["commodity"] == c]
    jitter = rng.uniform(-0.25, 0.25, len(g))
    ax.fill_between([-0.5, 2.5], 0, 0.9, color=viz.GRID, alpha=0.7, linewidth=0)
    ax.scatter(g["n_markets_within_50km"] + jitter, g["avg_price_index"], s=22,
               color=COMMODITY_COLORS[c], edgecolor="white", linewidth=0.6)
    t = g[g["is_price_trapped"]]
    ax.scatter(t["n_markets_within_50km"] + rng.uniform(-0.25, 0.25, len(t)), t["avg_price_index"], s=40,
               facecolor="none", edgecolor=INK, linewidth=1.2)
    ax.axhline(0.9, color=MUTED, linewidth=0.8)
    ax.set_title(f"{c}  ({int(g['is_price_trapped'].sum())} trapped)", loc="left", fontsize=10, color=INK_2)
    ax.set_xlabel("town locations with data within 50 km")
axes[0].set_ylabel("avg price index (district / state, same days)")
axes[0].set_ylim(0.4, 1.6)
viz.save(fig, "q4_district_access_scatter",
         note="Ringed dots are price-trapped. District centres from OpenStreetMap; town locations = distinct market coordinates.")
da[da["is_price_trapped"]]

,state,district,commodity,n_markets_within_50km,avg_price_index,opportunity_rate_pct,is_price_trapped,n_markets_within_50km_market_precision,is_price_trapped_market_precision,n_price_days
0,Gujarat,Junagarh,Onion,1,0.533,68.1,True,1,True,956
1,Madhya Pradesh,Guna,Potato,1,0.583,NaN,True,1,True,1412
2,Madhya Pradesh,Guna,Tomato,1,0.600,NaN,True,1,True,612
3,Maharashtra,Jalana,Potato,1,0.634,NaN,True,1,True,52
4,Madhya Pradesh,Tikamgarh,Onion,2,0.642,100.0,True,2,True,1
5,Maharashtra,Jalana,Onion,1,0.646,61.3,True,1,True,52
6,Maharashtra,Latur,Onion,1,0.666,80.0,True,1,True,5
7,Madhya Pradesh,Katni,Tomato,2,0.711,NaN,True,2,True,156
8,Gujarat,Jamnagar,Onion,1,0.753,100.0,True,1,True,792
9,Gujarat,Panchmahals,Tomato,1,0.756,80.0,True,1,True,5


## Robustness of all headline numbers (Q1-Q5)

In [4]:
rob = load("phase3", "robustness")
rob.pivot_table(index=["question", "metric", "commodity"], columns="version", values="value")

version                                             a_default  b_no_centroid  c_incl_suspect_low
question metric                          commodity                                              
Q1       median_abs_gap_rs_qtl           Onion        150.000        150.000             150.000
                                         Potato       100.000        100.000             100.000
                                         Tomato       200.000        200.000             200.000
         median_pct_gap                  Onion         10.500         10.600              10.500
                                         Potato        11.100         10.100              11.100
                                         Tomato        15.300         15.200              15.400
         p90_pct_gap                     Onion         55.000         57.100              55.500
                                         Potato        56.300         55.000              56.400
                                         Tomato        77.400         76.700              80.000
Q2       median_best_gain_rs_qtl_mid     All          383.000            NaN             385.000
                                         Onion        381.000            NaN             381.000
                                         Potato       290.000            NaN             291.000
                                         Tomato       473.000            NaN             476.000
         opportunity_rate_pct_high       All           35.000         29.600              35.100
                                         Onion         35.500         29.600              35.500
                                         Potato        24.800         20.500              24.800
                                         Tomato        44.600         38.100              44.800
         opportunity_rate_pct_low        All           49.900         43.800              50.000
                                         Onion         51.000         44.600              51.100
                                         Potato        40.100         34.200              40.100
                                         Tomato        58.500         52.100              58.700
         opportunity_rate_pct_mid        All           43.900         37.900              44.000
                                         Onion         44.800         38.400              44.800
                                         Potato        33.500         28.300              33.500
                                         Tomato        53.200         46.600              53.300
Q3       crash_rate_pct_all_months       Onion         15.420         16.080              15.480
                                         Potato         8.010          8.350               8.010
                                         Tomato        25.970         26.520              25.990
         crash_rate_pct_peak_month       Onion         42.050         43.440              42.050
                                         Potato        38.550         38.850              38.520
                                         Tomato        64.090         63.340              63.970
         peak_crash_month_number         Onion         12.000         12.000              12.000
                                         Potato        12.000         12.000              12.000
                                         Tomato        12.000         12.000              12.000
Q4       n_distinct_trapped_districts    All           22.000         26.000              22.000
         n_district_crop_total           All          463.000        463.000             463.000
         n_trapped_district_crop         All           29.000         36.000              29.000
                                         Onion         10.000         15.000              10.000
                                         Potato        10.000         11.000              10.000
                       